# 실습01_안경씌우기
1. 얼굴을 탐지한다.
2. 안경을 씌우기 위해 눈의 위치를 탐색한다.
3. 안경을 씌우는 위치에 안경에 해당하는 이미지를 사이즈 조정하여 넣는다. 비트 연산하여 넣는다.
4. 안경이 씌워진 얼굴 이미지를 출력한다.
5. 추가 : 눈의 각도를 계산해서 안경 이미지도 회전하는 코드를 작성해본다.

In [10]:
import cv2
import mediapipe as mp
import numpy as np

from pathlib import Path
import urllib.request
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [11]:
# TODO: 안경 PNG 불러오기
glasses_img = cv2.imread('data/glasses.png', cv2.IMREAD_UNCHANGED)

# 파일 로드 확인
if glasses_img is None:
    raise FileNotFoundError('glasses.png 파일을 찾을 수 없습니다.')

# 알파 채널 보정
if glasses_img.shape[2] == 3:
    alpha = np.full(glasses_img.shape[:2] + (1,), 255, dtype=np.uint8)
    glasses_img = np.concatenate([glasses_img, alpha], axis=2)

print('안경 이미지 크기:', glasses_img.shape)
cv2.imshow('glass_img', glasses_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

안경 이미지 크기: (353, 707, 4)


In [12]:
# FaceLandmarker 모델 준비
# Tasks API로 얼굴 메시 실행
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

FACE_LANDMARKER_MODEL = MODEL_DIR / 'face_landmarker.task'
FACE_LANDMARKER_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task'

In [13]:
def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return
    print(f'모델 다운로드 중: {model_path}')
    urllib.request.urlretrieve(model_url, model_path)


download_model(FACE_LANDMARKER_MODEL, FACE_LANDMARKER_MODEL_URL)

BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions

In [21]:
def create_face_landmarker(num_faces=1):
    # TODO: 얼굴 랜드마크 옵션 설정
    options = FaceLandmarkerOptions(
        base_options = BaseOptions(model_asset_path=str(FACE_LANDMARKER_MODEL)),
        running_mode = VisionRunningMode.VIDEO,
        num_faces = num_faces,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    return FaceLandmarker.create_from_options(options)


def to_mp_image(frame):
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)


def overlay_bgra(background, overlay, x, y):
    # BGRA 이미지 알파 블렌딩
    bg_h, bg_w = background.shape[:2]
    ov_h, ov_w = overlay.shape[:2]

    # 화면과 겹치는 영역 계산
    x1 = max(x, 0)
    y1 = max(y, 0)
    x2 = min(x + ov_w, bg_w)
    y2 = min(y + ov_h, bg_h)

    # TODO: 겹치는 영역 없으면 생략
    if x1 >= x2 or y1 >= y2:
        return background

    # overlay 겹침 영역 자르기
    ov_x1 = x1 - x
    ov_y1 = y1 - y
    ov_x2 = ov_x1 + (x2 - x1)
    ov_y2 = ov_y1 + (y2 - y1)

    overlay_crop = overlay[ov_y1:ov_y2, ov_x1:ov_x2]
    color = overlay_crop[:, :, :3].astype(np.float32)
    alpha = overlay_crop[:, :, 3:4].astype(np.float32) / 255.0

    roi = background[y1:y2, x1:x2].astype(np.float32)
    blended = alpha * color + (1.0 - alpha) * roi
    background[y1:y2, x1:x2] = blended.astype(np.uint8)
    return background

# 1.고정된 위치에 안경 올려보기

In [22]:
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        mp_image = to_mp_image(frame)
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)  # TODO: 얼굴 랜드마크 검출

        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                resized = cv2.resize(glasses_img, (200, 100), interpolation=cv2.INTER_AREA)  # TODO: 고정 좌표에 안경 합성
                overlay_bgra(frame, resized, 100, 100)

        cv2.imshow('Glasses Overlay', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# 2.눈의 위치를 계산하여 안경 씌어보기

In [26]:
def rotate_bgra(image, angle_degrees):
    # 안경 이미지 중심 회전
    h, w = image.shape[:2]
    center = (w / 2, h / 2)
    matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)

    cos = abs(matrix[0, 0])
    sin = abs(matrix[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    matrix[0, 2] += (new_w / 2) - center[0]
    matrix[1, 2] += (new_h / 2) - center[1]

    return cv2.warpAffine(
        image,
        matrix,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0),
    )

def eye_points(face_landmarks, frame_shape):
    # FaceLandmarker 좌표는 0~1 정규화
    frame_h, frame_w = frame_shape[:2]
    left_eye = face_landmarks[362]  # 왼쪽 눈 기준점
    right_eye = face_landmarks[133]  # 오른쪽 눈 기준점

    left = np.array([left_eye.x * frame_w, left_eye.y * frame_h])
    right = np.array([right_eye.x * frame_w, right_eye.y * frame_h])
    center = ((left + right) / 2).astype(int)
    distance = np.linalg.norm(left - right)
    angle = np.degrees(np.arctan2(left[1] - right[1], left[0] - right[0]))
    return left, right, center, distance, angle


def draw_glasses(frame, face_landmarks, rotate=False):
    # 눈 사이 거리 기준으로 안경 크기를 정합니다.
    _, _, center, eye_distance, angle = eye_points(face_landmarks, frame.shape)
    glasses_width = max(1, int(eye_distance * 5))

    # 회전을 고려하는 예제에서는 얼굴 기울기에 맞춰 안경 이미지를 먼저 회전합니다.
    source = rotate_bgra(glasses_img, -angle) if rotate else glasses_img
    scale = glasses_width / source.shape[1]
    glasses_height = max(1, int(source.shape[0] * scale))
    resized = cv2.resize(source, (glasses_width, glasses_height), interpolation=cv2.INTER_AREA)

    x = int(center[0] - glasses_width / 2)
    y = int(center[1] - glasses_height / 2)
    overlay_bgra(frame, resized, x, y)
    return frame

In [25]:
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        mp_image = to_mp_image(frame)
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)  # TODO: 얼굴 랜드마크 검출

        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                draw_glasses(frame, face_landmarks, rotate=False)  # TODO: 눈 거리/중심으로 안경 합성

        cv2.imshow('Glasses Overlay', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# 3.눈의 회전까지 고려해보기

In [27]:
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_face_landmarker(num_faces=1) as face_landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        mp_image = to_mp_image(frame)
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = face_landmarker.detect_for_video(mp_image, timestamp_ms)  # TODO: 얼굴 랜드마크 검출

        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                draw_glasses(frame, face_landmarks, rotate=True)  # TODO: 눈 거리/중심으로 안경 합성

        cv2.imshow('Glasses Overlay', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()